# Maker Intent Transformer / CUDA Training

Lチカのつづき の Maker Intent Encoder を Google Colab GPU で学習します。ランタイムは `GPU` を選んでください。

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('cuda device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 1. Repository Setup

GitHubに置いた場合は `REPO_URL` を入れてください。手元からアップロードする場合は Colab のファイル欄にこのリポジトリを置いて、次のセルを調整します。

In [ ]:
REPO_URL = ''  # 例: 'https://github.com/yourname/denshikousakuagent.git'
if REPO_URL:
    !git clone $REPO_URL
    %cd denshikousakuagent
else:
    print('REPO_URL is empty. Upload the repository folder, then run: %cd /content/denshikousakuagent')

In [ ]:
!pip install -q -r requirements-ml.txt

In [ ]:
!python -m ai_models.makergraph.generate_synthetic_intent_data \
  --output data/intent_training_synthetic.jsonl \
  --records-per-project 160

## 2. Train

小さい seed dataset でもパイプラインは通ります。本番では `data/intent_training_seed.jsonl` と同じ形式で制作ログ・失敗ログ・作品グラフを増やします。

In [ ]:
!python -m ai_models.makergraph.train_intent_encoder \
  --data data/intent_training_synthetic.jsonl \
  --output runs/maker_intent_transformer \
  --epochs 40 \
  --batch-size 5 \
  --d-model 256 \
  --n-heads 8 \
  --n-layers 4 \
  --embedding-dim 128 \
  --device cuda \
  --amp

## 3. Inference

In [ ]:
!python -m ai_models.makergraph.infer \
  --model-dir runs/maker_intent_transformer \
  --text '作りたいものは分からない。予算は5000円。かわいいものがいい。ESP32は持っている。' \
  --device cuda \
  --top-k 3

## 4. Evaluate Retrieval

In [ ]:
!python -m ai_models.makergraph.evaluate_retrieval \
  --model-dir runs/maker_intent_transformer \
  --data data/intent_training_synthetic.jsonl \
  --device cuda \
  --top-k 3

## 5. Train Project Graph Transformer

In [ ]:
!python -m ai_models.makergraph.train_project_graph_generator \
  --data data/intent_training_synthetic.jsonl \
  --output runs/project_graph_transformer \
  --epochs 40 \
  --batch-size 8 \
  --device cuda \
  --amp

In [ ]:
!python -m ai_models.makergraph.generate_project_graph \
  --model-dir runs/project_graph_transformer \
  --text '植物を枯らすので水やり通知を作りたい。自動ポンプは怖い。' \
  --device cuda

## 6. Save Checkpoints to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/makergraph_ai
!cp -r runs/maker_intent_transformer /content/drive/MyDrive/makergraph_ai/
!cp -r runs/project_graph_transformer /content/drive/MyDrive/makergraph_ai/